# Quickstart: recover a hidden descriptor

`jax-ibart` discovers compact, interpretable nonlinear descriptors. Here we hide
one inside synthetic data and recover it from scratch.

In [ ]:
import numpy as np
from jax_ibart import IBart

rng = np.random.default_rng(0)
n = 90
a, b, c, d = (rng.uniform(0.5, 2.0, n) for _ in range(4))
X = np.column_stack([a, b, c, d])
# the response depends only on the product a*b
y = 5.0 * (a * b) + rng.normal(0, 0.05, n)
names = ["a", "b", "c", "d"]
X.shape, y.shape

Fit with the differentiable backend (fast, no Java needed):

In [ ]:
model = IBart(backend="diff", schedule=("b",), max_dim=1, seed=0, steps=1200)
model.fit(X, y, feature_names=names)
model.descriptors_

The fitted symbolic model and its fit metrics:

In [ ]:
import numpy as np

pred = model.predict(X)
resid = y - pred
ss_res = float((resid ** 2).sum())
ss_tot = float(((y - y.mean()) ** 2).sum())
metrics = {
    "expression": str(model.expression()),
    "R2": 1 - ss_res / ss_tot,
    "RMSE": float(np.sqrt((resid ** 2).mean())),
    "MAE": float(np.abs(resid).mean()),
    "max_abs_error": float(np.abs(resid).max()),
}
for k, v in metrics.items():
    print(f"{k:>14}: {v}")

A parity plot (predicted vs. true) is the standard visual fit check:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4.5, 4.5))
lo, hi = float(min(y.min(), pred.min())), float(max(y.max(), pred.max()))
ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="parity")
ax.scatter(y, pred, s=18, alpha=0.7)
ax.set_xlabel("true y")
ax.set_ylabel("predicted y")
ax.set_title(f"Parity (R$^2$={metrics['R2']:.4f}, RMSE={metrics['RMSE']:.3f})")
ax.legend()
ax.set_aspect("equal")
fig.tight_layout()
plt.show()

It recovers `a*b` and fits `~5*a*b`, the hidden relationship. The descriptor is
*symbolic*, so you can read the physics straight off the model, and the parity
plot lands on the diagonal.